# Universal 3.5 Pro Sync — Speech-to-Text on Amazon SageMaker

This notebook demonstrates end-to-end usage of the **Universal 3.5 Pro Sync** speech recognition model package from AWS Marketplace: subscribing, deploying a real-time endpoint, transcribing audio, enabling word-level timestamps, running batch transform, and cleaning up.

The model transcribes speech audio and returns a transcript with per-word confidence scores and, optionally, word-level start/end times.

## Contents

1. [Prerequisites](#prereqs)
2. [Subscribe to the model package](#subscribe)
3. [Setup](#setup)
4. [Deploy a real-time endpoint](#deploy)
5. [Transcribe audio](#invoke)
6. [Word-level timestamps](#timestamps)
7. [Response format](#response)
8. [Configuration options](#config)
9. [Service limits](#limits)
10. [Batch transform](#batch)
11. [Clean up](#cleanup)
12. [Unsubscribe](#unsubscribe)


<a id='prereqs'></a>
## 1. Prerequisites

- An active subscription to **Universal 3.5 Pro Sync** in AWS Marketplace.
- An IAM role with `AmazonSageMakerFullAccess` (or equivalent permissions to create models, endpoints, and transform jobs).
- Sufficient service quota for the GPU instance type you choose.
- Run this notebook in the same AWS Region as your subscription.

<a id='subscribe'></a>
## 2. Subscribe to the model package

To subscribe to the model package:

1. Open the **Universal 3.5 Pro Sync** model package listing page in AWS Marketplace.
2. Read the product overview, pricing, and usage information, and review the End User License Agreement.
3. Choose **Continue to Subscribe**, then accept the offer.
4. After subscribing, open **AWS Marketplace > Manage subscriptions** (or the SageMaker console's **Marketplace model packages** page), select this product, and copy the **model package ARN** for your AWS Region.
5. Paste that ARN into the `MODEL_PACKAGE_ARN` variable in the deploy section below.


<a id='setup'></a>
## 3. Setup

In [ ]:
# Install the SageMaker Python SDK. Some environments (including recent
# SageMaker Distribution images) do not ship it preinstalled.
%pip install -q "sagemaker>=2.200,<3"


In [ ]:
import json
import time

import boto3
import sagemaker

session = sagemaker.Session()
region = session.boto_region_name
sm = boto3.client("sagemaker", region_name=region)
smr = boto3.client("sagemaker-runtime", region_name=region)

# The IAM role SageMaker assumes to pull the model. In SageMaker Studio or a
# notebook instance this is inferred; otherwise set it explicitly.
role = sagemaker.get_execution_role()

print(f"Region: {region}")
print(f"Role:   {role}")

<a id='deploy'></a>
## 4. Deploy a real-time endpoint

> ### ⚠️ Required: pin the inference AMI version
>
> This model **requires** `InferenceAmiVersion="al2-ami-sagemaker-inference-gpu-2-1"` on the endpoint
> configuration's production variant.
>
> That AMI provides NVIDIA driver 535 with CUDA-compat mounting disabled. Without it, the host mounts
> CUDA compatibility libraries that conflict with the container's CUDA runtime, and the container exits
> during startup with:
>
> ```
> RuntimeError: Unexpected error from cudaGetDeviceCount().
> Error 803: system has unsupported display driver / cuda driver combination
> ```
>
> The endpoint then fails its ping health check and settles in `Failed`. Because `InferenceAmiVersion`
> is a property of the endpoint configuration rather than the model package, it must be set by you at
> deploy time — it cannot be supplied by the model package itself.

Set the model package ARN for your Region from the AWS Marketplace listing page.

In [ ]:
# From the AWS Marketplace listing — use the ARN for your Region.
MODEL_PACKAGE_ARN = "<YOUR_MODEL_PACKAGE_ARN>"

MODEL_NAME = "universal-3-5-pro-sync"
ENDPOINT_CONFIG_NAME = "universal-3-5-pro-sync-config"
ENDPOINT_NAME = "universal-3-5-pro-sync-endpoint"

# Supported real-time instance types: ml.g6e.2xlarge, ml.g5.2xlarge, ml.g5.xlarge
INSTANCE_TYPE = "ml.g5.2xlarge"

# Required — see the note above.
INFERENCE_AMI_VERSION = "al2-ami-sagemaker-inference-gpu-2-1"

In [ ]:
sm.create_model(
    ModelName=MODEL_NAME,
    PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
    ExecutionRoleArn=role,
    EnableNetworkIsolation=True,  # required for AWS Marketplace model packages
)
print(f"Created model: {MODEL_NAME}")

In [ ]:
sm.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ProductionVariants=[
        {
            "VariantName": "primary",
            "ModelName": MODEL_NAME,
            "InitialInstanceCount": 1,
            "InstanceType": INSTANCE_TYPE,
            "InferenceAmiVersion": INFERENCE_AMI_VERSION,
            # Model weights take several minutes to load; allow ample startup time.
            "ContainerStartupHealthCheckTimeoutInSeconds": 1800,
        }
    ],
)

sm.create_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
)
print(f"Creating endpoint: {ENDPOINT_NAME}")

Endpoint creation typically takes 10–15 minutes: the container image is large and the model weights are
loaded onto the GPU at startup.

In [ ]:
waiter = sm.get_waiter("endpoint_in_service")
waiter.wait(EndpointName=ENDPOINT_NAME, WaiterConfig={"Delay": 30, "MaxAttempts": 80})

status = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"]
print(f"Endpoint status: {status}")

<a id='invoke'></a>
## 5. Transcribe audio

The simplest request sends raw audio bytes with a `Content-Type` of `audio/wav`.

This notebook uses the sample recordings in [`data/input/real-time/`](data/input/real-time) —
public-domain LibriVox audiobook clips at 16 kHz mono 16-bit PCM, the recommended format.
Substitute your own files as needed.


In [ ]:
AUDIO_PATH = "data/input/real-time/sample-10s.wav"

with open(AUDIO_PATH, "rb") as f:
    audio_bytes = f.read()

response = smr.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="audio/wav",
    Body=audio_bytes,
)

result = json.loads(response["Body"].read())
print(result["text"])

<a id='timestamps'></a>
## 6. Word-level timestamps

To pass options, send the request as `multipart/form-data` with two parts:

| Part | Content-Type | Contents |
|------|--------------|----------|
| `audio`  | `audio/wav` | the audio bytes |
| `config` | `application/json` | a JSON object of options |

Setting `timestamps` to `true` adds `start` and `end` (milliseconds from the beginning of the audio) to
each word. It defaults to `false`, so a request without a `config` part returns words with text and
confidence only.

The helper below builds the multipart body.

In [ ]:
import uuid


def build_multipart(audio_bytes: bytes, config: dict, filename: str = "audio.wav"):
    """Build a multipart/form-data body with `audio` and `config` parts.

    Returns (body_bytes, content_type).
    """
    boundary = f"----boundary{uuid.uuid4().hex}"
    parts = [
        (
            f"--{boundary}\r\n"
            f'Content-Disposition: form-data; name="audio"; filename="{filename}"\r\n'
            f"Content-Type: audio/wav\r\n\r\n"
        ).encode(),
        audio_bytes,
        (
            f"\r\n--{boundary}\r\n"
            f'Content-Disposition: form-data; name="config"\r\n'
            f"Content-Type: application/json\r\n\r\n"
            f"{json.dumps(config)}\r\n"
            f"--{boundary}--\r\n"
        ).encode(),
    ]
    return b"".join(parts), f"multipart/form-data; boundary={boundary}"


def transcribe(audio_bytes: bytes, config: dict | None = None) -> dict:
    """Transcribe audio, optionally with configuration options."""
    if config is None:
        body, content_type = audio_bytes, "audio/wav"
    else:
        body, content_type = build_multipart(audio_bytes, config)

    response = smr.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType=content_type,
        Body=body,
    )
    return json.loads(response["Body"].read())

In [ ]:
result = transcribe(audio_bytes, {"timestamps": True})

print(result["text"])
print()
print(f"{'start':>8} {'end':>8}  word")
for word in result["words"][:15]:
    print(f"{word['start']:>8} {word['end']:>8}  {word['text']}")

<a id='response'></a>
## 7. Response format

```json
{
  "text": "Smoke from hundreds of wildfires in Canada is triggering air quality alerts...",
  "words": [
    {"text": "Smoke", "confidence": 0.9957, "start": 32, "end": 449}
  ],
  "confidence": 0.982,
  "audio_duration_ms": 60000,
  "session_id": "65d9eec9-bc37-4544-bf9a-4402e26ac7c0",
  "request_time_ms": 2368.4
}
```

| Field | Description |
|-------|-------------|
| `text` | Full transcript, formatted and punctuated |
| `words[].text` | The word as it appears in the transcript |
| `words[].confidence` | Model confidence for that word, 0–1 |
| `words[].start`, `words[].end` | Milliseconds from the start of the audio. Present only when `timestamps` is `true`. Omitted for any word that cannot be aligned — timings are exact or absent, never estimated. |
| `confidence` | Mean confidence across the transcript |
| `audio_duration_ms` | Duration of the submitted audio |
| `session_id` | Unique identifier for the request |
| `request_time_ms` | Server-side processing time |

<a id='config'></a>
## 8. Configuration options

Options are passed in the `config` part as a JSON object.

| Option | Type | Description |
|--------|------|-------------|
| `timestamps` | boolean | Include word-level `start`/`end` times. Default `false`. |
| `sample_rate` | integer | Sample rate of the audio, in Hz. |
| `channels` | integer | Number of audio channels. |
| `prompt` | string | Context to bias recognition. |
| `keyterms` | list | Terms to boost. Mutually exclusive with `keyterms_prompt` and `word_boost`. |

Unrecognized options are rejected with HTTP 400 and a message naming the offending field, so typos fail
loudly rather than being silently ignored.

Supported request content types: `audio/wav`, `audio/pcm`. Responses are `application/json`.

<a id='limits'></a>
## 9. Service limits

| Limit | Real-time endpoint | Batch transform |
|-------|-------------------|-----------------|
| Maximum audio duration per request | 120 seconds | 120 seconds |
| Maximum payload size | 25 MB | 100 MB |
| Maximum processing time per request | 60 seconds | 60 minutes |

Audio longer than 120 seconds is rejected with HTTP 413. Split longer recordings into segments before
submitting them.

This model runs in network isolation, as all AWS Marketplace model packages do, and therefore cannot be
deployed to SageMaker Serverless Inference endpoints.

<a id='batch'></a>
## 10. Batch transform

Batch transform processes many audio files from Amazon S3 in a single job. Each object is sent to the
model as one request, and the transcript is written to the output prefix as `<filename>.out`.

> **Note:** Batch transform sends each object as a raw request body, so the `config` part cannot be
> supplied. Transcripts from batch transform therefore contain word text and confidence but **not**
> timestamps. Use a real-time endpoint when you need word-level timings.

Set `SplitType` to `None` so each file is treated as a single record rather than being split.

In [ ]:
# Upload the bundled batch input files to S3.
S3_INPUT = session.upload_data("data/input/batch", key_prefix="universal-3-5-pro-sync/batch-input")
S3_OUTPUT = f"s3://{session.default_bucket()}/universal-3-5-pro-sync/batch-output"
TRANSFORM_JOB_NAME = f"universal-3-5-pro-sync-{int(time.time())}"

sm.create_transform_job(
    TransformJobName=TRANSFORM_JOB_NAME,
    ModelName=MODEL_NAME,
    MaxConcurrentTransforms=1,
    MaxPayloadInMB=40,
    TransformInput={
        "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": S3_INPUT}},
        "ContentType": "audio/wav",
        "CompressionType": "None",
        "SplitType": "None",
    },
    TransformOutput={
        "S3OutputPath": S3_OUTPUT,
        "Accept": "application/json",
        "AssembleWith": "None",
    },
    TransformResources={
        "InstanceType": "ml.g5.2xlarge",
        "InstanceCount": 1,
    },
)

sm.get_waiter("transform_job_completed_or_stopped").wait(TransformJobName=TRANSFORM_JOB_NAME)
print(sm.describe_transform_job(TransformJobName=TRANSFORM_JOB_NAME)["TransformJobStatus"])

In [ ]:
# Display the batch transform outputs (one <filename>.out per input file).
import os

s3 = boto3.client("s3", region_name=region)
bucket = session.default_bucket()

for obj in s3.list_objects_v2(Bucket=bucket, Prefix="universal-3-5-pro-sync/batch-output").get("Contents", []):
    body = s3.get_object(Bucket=bucket, Key=obj["Key"])["Body"].read()
    result = json.loads(body)
    print(f"{os.path.basename(obj['Key'])}: {result['text'][:120]}")


<a id='cleanup'></a>
## 11. Clean up

Endpoints bill for as long as they are running. Delete the endpoint when you are finished.

Deleting these resources does not cancel your AWS Marketplace subscription; manage that from the
**Machine Learning subscriptions** page in the SageMaker console.

In [ ]:
sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_CONFIG_NAME)
sm.delete_model(ModelName=MODEL_NAME)
print("Deleted endpoint, endpoint configuration, and model.")

<a id='unsubscribe'></a>
## 12. Unsubscribe (optional)

If you no longer need the model, cancel the subscription so you are not charged for future use:

1. Confirm no deployable model, live endpoint, or running batch transform job created from this
   model package remains in your account.
2. Open **Your Marketplace Software** on the AWS Marketplace website (or **AWS Marketplace >
   Manage subscriptions**), locate this product, and cancel the subscription.
